# Model Optimization: WANDA Pruning

In this notebook, we'll apply WANDA (Weight ANalysis for Deep leArning) pruning techniques to our models using distributed processing. This is an advanced pruning technique that considers both weight magnitudes and activation statistics when deciding which weights to prune.

## What is WANDA Pruning?

WANDA pruning is an advanced technique that improves upon traditional magnitude-based pruning methods by incorporating activation statistics. While standard pruning methods like L1 unstructured pruning only look at the absolute values of weights, WANDA considers how those weights interact with activations during inference.

### Key Differences from Standard Pruning:
- **Activation-Aware**: Considers both weight magnitudes and activation statistics
- **Better Accuracy Preservation**: Tends to maintain model accuracy better than simple magnitude pruning
- **Calibration Data**: Uses a small set of sample inputs to collect activation statistics
- **Importance Scoring**: Calculates importance as weight magnitude × activation magnitude

### Benefits of WANDA Pruning:
- **Better Performance**: Often achieves better accuracy-sparsity tradeoff
- **More Intelligent Selection**: Preserves weights that have high impact on outputs
- **Adaptability**: Adapts to the specific characteristics of the model and task
- **Reduced Fine-tuning Needs**: Often requires less fine-tuning after pruning

### Distributed Processing Approach
Like in the previous notebook, we'll use SageMaker Processing jobs to perform pruning on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Model Information

In [ ]:
# Try to load baseline metrics if they exist
try:
    with open('baseline_metrics.json', 'r') as f:
        baseline_metrics = json.load(f)
    print(f"Loaded baseline metrics for {len(baseline_metrics)} models")
except FileNotFoundError:
    print("baseline_metrics.json not found. Will proceed without baseline metrics.")
    baseline_metrics = {}

# Try to load pruned metrics if they exist (for comparison)
try:
    with open('pruned_metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned_metrics.json not found. Will proceed without pruned metrics for comparison.")
    pruned_metrics = {}

# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment-analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question-answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked-lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create WANDA Pruning Scripts

In this section, we'll create the necessary scripts for WANDA pruning. This includes:
1. A requirements.txt file with the necessary dependencies
2. A Python script that implements the WANDA pruning algorithm

### What is WANDA Pruning?
WANDA (Weight ANalysis for Deep leArning) is an advanced pruning technique that considers both weight magnitudes and activation statistics when deciding which weights to prune. This approach tends to preserve model accuracy better than simple magnitude-based pruning.

In [ ]:
# Check if the wanda_pruning_scripts directory exists, if not create it
import os
if not os.path.exists('wanda_pruning_scripts'):
    os.makedirs('wanda_pruning_scripts')
    print("Created wanda_pruning_scripts directory")
else:
    print("wanda_pruning_scripts directory already exists")

## 6. Launch SageMaker Processing Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform WANDA pruning. Each model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="wanda-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch WANDA pruning jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing WANDA pruning jobs for all models...")

for model_key in model_info.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}_wanda_pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='wanda_pruned_model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-amount', '0.3',
            '--calibration-samples', '32'
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all WANDA pruning jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"wanda-pruning-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='wanda_pruning_script.py',
            source_dir='wanda_pruning_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    for job_name in job_names:
        job = ProcessingJob.from_processing_name(job_name, sagemaker_session=sagemaker_session)
        status = job.describe()['ProcessingJobStatus']
        job_statuses[job_name] = status
        
        if status in ['InProgress', 'Stopping']:
            all_complete = False
    
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 7. Collect Results

Now that the WANDA pruning jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the pruned model, such as size, inference time, and pruning parameters.

In [ ]:
# Download and combine results
wanda_pruned_metrics = {}
sagemaker_client = boto3.client("sagemaker")

for model_key in model_info.keys():
    # Check if we have a job for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}_wanda_pruned/wanda_pruned_metrics.json',
            f'temp_{model_key}_wanda_pruned_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_wanda_pruned_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        wanda_pruned_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('wanda_pruned_metrics.json', 'w') as f:
    json.dump(wanda_pruned_metrics, f, indent=2)

print(f"\nSaved WANDA pruned metrics for {len(wanda_pruned_metrics)} models to wanda_pruned_metrics.json")

## 8. Compare Results

Now we'll compare the performance of the WANDA pruned models against the baseline models and the standard pruned models. This comparison helps us understand the benefits of WANDA pruning over traditional pruning methods.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in wanda_pruned_metrics.keys():
    baseline = baseline_metrics.get(model_key, {})
    standard_pruned = pruned_metrics.get(model_key, {})
    wanda_pruned = wanda_pruned_metrics[model_key]
    
    # Prepare data for this model
    model_data = {
        'Model': wanda_pruned['model_name'],
        'WANDA Pruning Amount': f"{wanda_pruned['pruning_amount'] * 100:.1f}%"
    }
    
    # Add baseline metrics if available
    if baseline:
        model_data.update({
            'Baseline Size (MB)': baseline.get('model_size', 'N/A'),
            'Baseline Inference (ms)': baseline.get('inference_time', 'N/A')
        })
    
    # Add standard pruned metrics if available
    if standard_pruned:
        model_data.update({
            'Standard Pruned Size (MB)': standard_pruned.get('model_size', 'N/A'),
            'Standard Pruned Inference (ms)': standard_pruned.get('inference_time', 'N/A'),
            'Standard Pruning Method': standard_pruned.get('pruning_method', 'N/A'),
            'Standard Pruning Amount': f"{standard_pruned.get('pruning_amount', 0) * 100:.1f}%",
            'Standard Size Reduction (%)': standard_pruned.get('size_reduction', 'N/A'),
            'Standard Time Improvement (%)': standard_pruned.get('time_improvement', 'N/A'),
            'Standard Sparsity (%)': standard_pruned.get('sparsity', 'N/A')
        })
    
    # Add WANDA pruned metrics
    model_data.update({
        'WANDA Pruned Size (MB)': wanda_pruned['model_size'],
        'WANDA Pruned Inference (ms)': wanda_pruned['inference_time'],
        'WANDA Size Reduction (%)': wanda_pruned['size_reduction'],
        'WANDA Time Improvement (%)': wanda_pruned['time_improvement'],
        'WANDA Sparsity (%)': wanda_pruned['sparsity']
    })
    
    # Add comparison between WANDA and standard pruning if both are available
    if standard_pruned:
        # Calculate improvements of WANDA over standard pruning
        wanda_vs_standard_size = ((standard_pruned.get('model_size', 0) - wanda_pruned['model_size']) / 
                                 standard_pruned.get('model_size', 1) * 100)
        wanda_vs_standard_time = ((standard_pruned.get('inference_time', 0) - wanda_pruned['inference_time']) / 
                                 standard_pruned.get('inference_time', 1) * 100)
        
        model_data.update({
            'WANDA vs Standard Size Improvement (%)': wanda_vs_standard_size if wanda_vs_standard_size != float('inf') else 'N/A',
            'WANDA vs Standard Time Improvement (%)': wanda_vs_standard_time if wanda_vs_standard_time != float('inf') else 'N/A'
        })
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Next Steps

Now that we've applied both standard pruning and WANDA pruning to our models and compared their performance, we'll explore knowledge distillation in the next notebook to create even smaller, faster models.

### What We've Learned:
- How to apply WANDA pruning to transformer models for improved accuracy-sparsity tradeoff
- How to use activation statistics to make more intelligent pruning decisions
- How WANDA pruning compares to standard magnitude-based pruning
- How to use SageMaker Processing for distributed optimization tasks
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - Knowledge Distillation:
Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. This allows us to create models that are much smaller and faster while retaining most of the accuracy of the original model. The combination of pruning and knowledge distillation can lead to even greater size reductions and performance improvements.